# Cellular Automata — Recreating paper Tables 2, 3, 5–11 and verifying Table 12

Bhuvaneswari A & Bhattacharjee K, 2026 (arXiv:2603.19656).  This
notebook **re-runs the authors' constructions** using the production
`regpoly_cpp` C++ kernel (matricial-DE equidistribution, primitivity,
characteristic polynomial), reproducing what the paper itself computes
for:

- **Table 2** — component CA characteristics (`k`, rule-150 positions,
  `N1`, `N1/k`) for `k = 29..128`;
- **Table 3** — combined `(31, 32)` per-`t` equidistribution at `s = 1`;
- **Table 5** — four hand-picked `(k1, k2)` pairs with `N1` near half-degree;
- **Table 6** — enumeration of all `(k1, k2)` with `N1/k` not near 0.5;
- **Tables 7, 8, 9, 10** — combined `(31, 32)` at `s ∈ {2, 4, 7, 8}`;
- **Table 11** — Adak-Das `CA(90′)`/`CA(150′)` combinations close to 32, 64, 128.

For these eight tables the notebook **does not compare to the paper's
printed values** — it simply runs the construction the authors describe.

For **Table 12** the notebook verifies the two columns the user asked
about — *Period (≈ ρ)* and *Equidistribution* — against the paper's
claims.  Under the standard L'Écuyer / Proposition 2 convention this
kernel implements (`t · ℓ` rows, equi iff `rank = t · ℓ`) several of
the paper's ME claims do not hold; the disagreement is surfaced as a
separate column rather than analysed.

## 0. Imports, catalog, helpers

In [1]:
import math
import json
import time
from pathlib import Path

from regpoly.core.generator import Generator
from regpoly.core.combination import Combination
from regpoly.analyses.equidistribution_test import EquidistributionTest

WORD = 64
S_RANGE = list(range(2, 11))      # paper tests s ∈ {2, …, 10}

# Catalog: Cattell-Zhang rule-150 positions (0-indexed) for k = 29..128,
# and the Sophie-Germain prime k-lists for Adak-Das CA(90′) / CA(150′).
_DATA_PATH = Path('../../../packages/regpoly/src/regpoly/data/cellular_automata.json').resolve()
_CATALOG = json.loads(_DATA_PATH.read_text())
CZ = _CATALOG['cattell_zhang_1995']
ADAK_K = _CATALOG['adak_das_2021_CA90prime_k_values']  # same list for both modes
print(f'Cattell-Zhang catalog: {len(CZ)} entries, k ∈ [{min(int(k) for k in CZ)}, {max(int(k) for k in CZ)}]')
print(f'Adak-Das k values:     {ADAK_K}')

Cattell-Zhang catalog: 100 entries, k ∈ [29, 128]
Adak-Das k values:     [26, 29, 35, 39, 65, 69, 105, 113, 119]


In [2]:
def _phi(k_g, L_word):
    """Φ₁ ∪ Φ₂ — the set of `t` values to test per Proposition 4."""
    r_sqrt = int(math.isqrt(k_g))
    m = max(2, k_g // L_word)
    phi1 = set(range(m, r_sqrt + 1))
    phi2 = {k_g // l for l in range(1, r_sqrt + 1)} - set(range(m))
    return sorted(phi1 | phi2)


def delta_to_lambda(delta, k_g, L):
    """Convert Δ_ℓ (per resolution, length L+1) to Λ_t (per dimension, length k_g+1).

    Mirrors the routine in `test_cellular_automata_paper_disagreement.py`.
    """
    t_at_resolution = [0] * (L + 1)
    for l in range(1, L + 1):
        t_at_resolution[l] = (k_g // l) - max(0, delta[l])
    lam = [0] * (k_g + 1)
    for t in range(1, k_g + 1):
        l_star = min(L, k_g // t)
        if l_star == 0:
            continue
        l_achieved = 0
        for l in range(l_star, 0, -1):
            if t_at_resolution[l] >= t:
                l_achieved = l
                break
        lam[t] = l_star - l_achieved
    return lam


def _make_single(k, positions, s):
    return Generator.create('CellularAutomataGen', L=min(k, WORD),
                            k=k, rule150_positions=positions, s=s)


def _make_combined(k1, p1, k2, p2, s):
    g1 = _make_single(k1, p1, s)
    g2 = _make_single(k2, p2, s)
    comb = Combination.CreateFromFiles([[g1], [g2]], Lmax=WORD,
                                       temperings=[[], []])
    next(iter(comb))
    return g1, g2, comb


def _n1(gen):
    """Number of nonzero coefficients of the characteristic polynomial.

    `gen.char_poly()` returns a k-bit BitVect for the polynomial of degree
    k (the leading x^k term is implicit and always 1 for a primitive
    polynomial), so N1 = popcount(stored bits) + 1.
    """
    return bin(int(gen.char_poly()._val)).count('1') + 1


def _run_equi(comb):
    test = EquidistributionTest(L=WORD, delta=[10**9] * (WORD + 1),
                                mse=10**9, method=None)
    return test.run(comb)


print('helpers loaded')

helpers loaded


## Table 2 — Component CA characteristics (k = 29..128)

For each `k`, the paper lists the rule-150 cell positions (1-indexed in
the paper, 0-indexed in the catalog), `k/2`, `N1` (number of non-zero
coefficients of the characteristic polynomial), and `N1/k`.

In [3]:
print(f'{"k":>4} {"rule150 (1-indexed)":>22} {"k/2":>6} {"N1":>4} {"N1/k":>6}')
print('-' * 50)
rows_table2 = []
for k in range(29, 129):
    positions = CZ[str(k)]
    gen = _make_single(k, positions, s=1)
    n1 = _n1(gen)
    pos_1idx = ','.join(str(p + 1) for p in positions)
    print(f'{k:>4} {pos_1idx:>22} {k/2:>6.1f} {n1:>4} {n1/k:>6.2f}')
    rows_table2.append((k, positions, n1, n1/k))

print(f'\n{len(rows_table2)} rows')
near_half = [r for r in rows_table2 if abs(r[3] - 0.5) < 0.05]
print(f'CAs with N1/k within 0.05 of 0.5: {[r[0] for r in near_half]}')

   k    rule150 (1-indexed)    k/2   N1   N1/k
--------------------------------------------------
  29                      1   14.5   11   0.38
  30                      1   15.0    9   0.30
  31                     11   15.5    5   0.16
  32                   1,15   16.0   11   0.34
  33                      1   16.5   11   0.33
  34                   1,19   17.0   11   0.32
  35                      1   17.5   13   0.37
  36                      6   18.0   19   0.53
  37                      9   18.5   17   0.46
  38                      7   19.0   13   0.34
  39                      1   19.5   13   0.33
  40                      8   20.0   17   0.42
  41                      1   20.5   19   0.46
  42                     19   21.0   23   0.55
  43                      3   21.5   17   0.40
  44                   4,26   22.0   21   0.48
  45                      9   22.5   17   0.38
  46                   2,10   23.0   19   0.41
  47                     13   23.5    9   0.19
  48     

## Tables 3, 7, 8, 9, 10 — Per-`t` equidistribution for combined (31, 32) at `s ∈ {1, 2, 4, 7, 8}`

For each `t ∈ Φ₁ ∪ Φ₂`, builds the combined CA, runs the matricial-DE
equidistribution test, and prints the per-`t` `(ℓ*_t, ℓ_t, t·ℓ*_t, rank)`
verdict.  Rule-150 positions are read from the catalog: component 1 of
size 31 has rule 150 at cell 11 (1-indexed → catalog cell `10`);
component 2 of size 32 has rule 150 at cells 1 and 15 (catalog cells
`[0, 14]`).

In [4]:
def run_per_t(k1, p1, k2, p2, s):
    _, _, comb = _make_combined(k1, p1, k2, p2, s)
    res = _run_equi(comb)
    lam = delta_to_lambda(res.ecart, comb.k_g, comb.L)
    print(f'\nCombined ({k1}, {k2}) at s = {s}    (k_g = {comb.k_g}, L = {comb.L})')
    print(f'{"t":>4} {"l*":>4} {"l_t":>4} {"lambda_t":>9} {"verdict":>10}')
    print('-' * 38)
    PHI = _phi(comb.k_g, comb.L)
    n_equi = 0
    for t in PHI:
        l_star = min(comb.L, comb.k_g // t)
        l_t = l_star - lam[t]
        equi = (l_t == l_star)
        n_equi += int(equi)
        print(f'{t:>4} {l_star:>4} {l_t:>4} {lam[t]:>9}  {"equi" if equi else "NOT equi":>9}')
    print(f'\nME (all rows equi)? {n_equi == len(PHI)}   ({n_equi}/{len(PHI)})')


P1, P2 = CZ['31'], CZ['32']
for s in [1, 2, 4, 7, 8]:
    run_per_t(31, P1, 32, P2, s)


Combined (31, 32) at s = 1    (k_g = 63, L = 31)
   t   l*  l_t  lambda_t    verdict
--------------------------------------
   2   31    2        29   NOT equi
   3   21    2        19   NOT equi
   4   15    2        13   NOT equi
   5   12    2        10   NOT equi
   6   10    2         8   NOT equi
   7    9    2         7   NOT equi
   9    7    2         5   NOT equi
  10    6    2         4   NOT equi
  12    5    2         3   NOT equi
  15    4    2         2   NOT equi
  21    3    2         1   NOT equi
  31    2    1         1   NOT equi
  63    1    1         0       equi

ME (all rows equi)? False   (1/13)

Combined (31, 32) at s = 2    (k_g = 63, L = 31)
   t   l*  l_t  lambda_t    verdict
--------------------------------------
   2   31    4        27   NOT equi
   3   21    4        17   NOT equi
   4   15    4        11   NOT equi
   5   12    4         8   NOT equi
   6   10    4         6   NOT equi
   7    9    4         5   NOT equi
   9    7    4         3   NOT

## Table 5 — four `(k1, k2)` pairs with `N1` near half-degree

For each pair and each `s ∈ {2, …, 10}`, compute:

- `s_period`: the `s` values where both component CAs remain primitive at
  spacing `s` *and* the two component periods are coprime — i.e. where the
  combined PRNG attains close-to-maximal period;
- `s_ME`: the `s` values where the kernel's standard convention reports
  the combined CA to be maximally equidistributed;
- `ρ_ME`: their intersection (paper's `ρ ME` column).

In [5]:
def enum_pair(k1, k2, S=S_RANGE):
    p1, p2 = CZ[str(k1)], CZ[str(k2)]
    coprime_periods = math.gcd(2**k1 - 1, 2**k2 - 1) == 1
    s_period, s_me = [], []
    for s in S:
        g1, g2, comb = _make_combined(k1, p1, k2, p2, s)
        period_ok = (coprime_periods
                     and g1._cpp_gen.is_full_period()
                     and g2._cpp_gen.is_full_period())
        me_ok = _run_equi(comb).is_me()
        if period_ok: s_period.append(s)
        if me_ok:     s_me.append(s)
    rho_me = [s for s in s_period if s in s_me]
    return s_period, s_me, rho_me


TABLE5_PAIRS = [(37, 42), (37, 44), (37, 50), (37, 92)]
print(f'{"k1":>4} {"k2":>4}  {"s_period":>20}  {"s_ME":>20}  {"rho_ME":>20}')
print('-' * 80)
for (k1, k2) in TABLE5_PAIRS:
    s_p, s_me, rho_me = enum_pair(k1, k2)
    print(f'{k1:>4} {k2:>4}  {str(s_p):>20}  {str(s_me):>20}  {str(rho_me):>20}')

  k1   k2              s_period                  s_ME                rho_ME
--------------------------------------------------------------------------------
  37   42      [2, 4, 5, 8, 10]                    []                    []
  37   44          [2, 4, 7, 8]                    []                    []
  37   50   [2, 4, 5, 7, 8, 10]                    []                    []


  37   92          [2, 4, 7, 8]                    []                    []


## Table 6 — Enumerate all `(k1, k2)` with `N1/k` *not* near 0.5

Iterate over all unordered pairs `(k1, k2)` with `31 ≤ k1 < k2 ≤ MAX_K`,
both components excluded from the half-degree set, and components whose
underlying periods `2^{k_i} − 1` are pairwise coprime.  For each, list
`s_period`, `s_ME`, `ρ_ME` — same columns as Table 5.

`MAX_K = 64` covers the part of Table 6 visible in the paper's left
column.  Set `MAX_K = 128` to enumerate the paper's full range (~4 min
of wall-clock time on a modern laptop).

In [6]:
MAX_K = 64           # bump to 128 for the paper's full range
HALF_DEGREE = {37, 42, 44, 50, 92}

candidates = []
for k1 in range(31, MAX_K + 1):
    if k1 in HALF_DEGREE:
        continue
    for k2 in range(k1 + 1, MAX_K + 1):
        if k2 in HALF_DEGREE:
            continue
        if math.gcd(2**k1 - 1, 2**k2 - 1) != 1:
            continue
        candidates.append((k1, k2))

print(f'{len(candidates)} candidate (k1, k2) pairs in range [31, {MAX_K}]\n')

t0 = time.perf_counter()
print(f'{"k1":>4} {"k2":>4}  {"s_period":>26}  {"s_ME":>16}  {"rho_ME":>16}')
print('-' * 76)
n_period = 0
n_me = 0
for (k1, k2) in candidates:
    s_p, s_me, rho_me = enum_pair(k1, k2)
    if s_p:
        n_period += 1
    if s_me:
        n_me += 1
    if s_p:                       # always print rows with at least one valid spacing
        print(f'{k1:>4} {k2:>4}  {str(s_p):>26}  {str(s_me):>16}  {str(rho_me):>16}')
elapsed = time.perf_counter() - t0
print(f'\n{n_period}/{len(candidates)} pairs with at least one s achieving close-to-maximal period')
print(f'{n_me}/{len(candidates)} pairs with at least one s achieving ME under the kernel convention')
print(f'elapsed {elapsed:.1f} s')

286 candidate (k1, k2) pairs in range [31, 64]

  k1   k2                    s_period              s_ME            rho_ME
----------------------------------------------------------------------------
  31   32                [2, 4, 7, 8]                []                []
  31   33   [2, 3, 4, 5, 6, 8, 9, 10]                []                []
  31   34         [2, 4, 5, 7, 8, 10]                []                []
  31   35  [2, 3, 4, 5, 6, 7, 8, 9, 10]                []                []
  31   36                   [2, 4, 8]                []                []
  31   38         [2, 4, 5, 7, 8, 10]                []                []
  31   39   [2, 3, 4, 5, 6, 8, 9, 10]                []                []
  31   40                [2, 4, 7, 8]                []                []


  31   41  [2, 3, 4, 5, 6, 7, 8, 9, 10]                []                []
  31   43  [2, 3, 4, 5, 6, 7, 8, 9, 10]                []                []
  31   45   [2, 3, 4, 5, 6, 8, 9, 10]                []                []


  31   46         [2, 4, 5, 7, 8, 10]                []                []
  31   47  [2, 3, 4, 5, 6, 7, 8, 9, 10]                []                []
  31   48                   [2, 4, 8]                []                []
  31   49  [2, 3, 4, 5, 6, 7, 8, 9, 10]                []                []
  31   51   [2, 3, 4, 5, 6, 8, 9, 10]                []                []
  31   52                [2, 4, 7, 8]                []                []


  31   53  [2, 3, 4, 5, 6, 7, 8, 9, 10]                []                []
  31   54            [2, 4, 5, 8, 10]                []                []
  31   55  [2, 3, 4, 5, 6, 7, 8, 9, 10]                []                []
  31   56                [2, 4, 7, 8]                []                []
  31   57   [2, 3, 4, 5, 6, 8, 9, 10]                []                []
  31   58         [2, 4, 5, 7, 8, 10]                []                []


  31   59  [2, 3, 4, 5, 6, 7, 8, 9, 10]                []                []
  31   60                   [2, 4, 8]                []                []
  31   61  [2, 3, 4, 5, 6, 7, 8, 9, 10]                []                []
  31   63   [2, 3, 4, 5, 6, 8, 9, 10]                []                []
  31   64                [2, 4, 7, 8]                []                []
  32   33                   [2, 4, 8]                []                []
  32   35                [2, 4, 7, 8]                []                []
  32   39                   [2, 4, 8]                []                []
  32   41                [2, 4, 7, 8]                []                []
  32   43                [2, 4, 7, 8]                []                []
  32   45                   [2, 4, 8]                []                []
  32   47                [2, 4, 7, 8]                []                []


...

  59   60                   [2, 4, 8]                []                []
  59   61  [2, 3, 4, 5, 6, 7, 8, 9, 10]                []                []


  59   62         [2, 4, 5, 7, 8, 10]                []                []
  59   63   [2, 3, 4, 5, 6, 8, 9, 10]                []                []


  59   64                [2, 4, 7, 8]                []                []
  60   61                   [2, 4, 8]                []                []


  61   62         [2, 4, 5, 7, 8, 10]                []                []
  61   63   [2, 3, 4, 5, 6, 8, 9, 10]                []                []
  61   64                [2, 4, 7, 8]                []                []


  62   63            [2, 4, 5, 8, 10]                []                []
  63   64                   [2, 4, 8]                []                []

286/286 pairs with at least one s achieving close-to-maximal period
0/286 pairs with at least one s achieving ME under the kernel convention
elapsed 19.2 s


## Table 11 — Adak-Das `CA(90′)` / `CA(150′)` combinations

For Sophie-Germain prime sizes `k` the Adak-Das constructions give
maximal-length CAs with rule-150 either only at cell 0 (`CA(90′)`) or
everywhere except cell 0 (`CA(150′)`).  The paper groups them by
size band — "close to 32", "close to 64", "close to 128" — and
considers the three pairings `(90′, 90′)`, `(150′, 150′)`, `(90′, 150′)`
for ordered `(k1, k2)` with `k1 ≠ k2`.

Set `INCLUDE_LARGE = True` to include `k ≥ 105` (~1 min extra).

In [7]:
INCLUDE_LARGE = True

def adak_pos(k, mode):
    """mode='90p' → rule150 only at cell 0; mode='150p' → rule150 everywhere
    except cell 0."""
    return [0] if mode == '90p' else list(range(1, k))


def enum_adak_pair(k1, mode1, k2, mode2, S=S_RANGE):
    p1, p2 = adak_pos(k1, mode1), adak_pos(k2, mode2)
    coprime_periods = math.gcd(2**k1 - 1, 2**k2 - 1) == 1
    s_period, s_me = [], []
    for s in S:
        g1, g2, comb = _make_combined(k1, p1, k2, p2, s)
        period_ok = (coprime_periods
                     and g1._cpp_gen.is_full_period()
                     and g2._cpp_gen.is_full_period())
        me_ok = _run_equi(comb).is_me()
        if period_ok: s_period.append(s)
        if me_ok:     s_me.append(s)
    rho_me = [s for s in s_period if s in s_me]
    return s_period, s_me, rho_me


GROUPS = [
    ('close to 32',  [26, 29, 35, 39]),
    ('close to 64',  [65, 69]),
]
if INCLUDE_LARGE:
    GROUPS.append(('close to 128', [105, 113, 119]))

PAIRINGS = [('90p', '90p'), ('150p', '150p'), ('90p', '150p')]

for (label, ks) in GROUPS:
    print(f'\n=== Group: {label}  (k ∈ {ks}) ===\n')
    for (m1, m2) in PAIRINGS:
        print(f'-- pairing ({m1}, {m2}) --')
        any_row = False
        for k1 in ks:
            for k2 in ks:
                if k1 == k2:
                    continue
                s_p, s_me, rho_me = enum_adak_pair(k1, m1, k2, m2)
                if s_p:                    # print rows that achieve close-to-max period at some s
                    any_row = True
                    print(f'   ({k1:>3}, {k2:>3})  s_period={s_p}  s_ME={s_me}  rho_ME={rho_me}')
        if not any_row:
            print('   (no pair with close-to-maximal period at any s)')


=== Group: close to 32  (k ∈ [26, 29, 35, 39]) ===

-- pairing (90p, 90p) --
   ( 26,  29)  s_period=[2, 4, 5, 7, 8, 10]  s_ME=[]  rho_ME=[]
   ( 26,  35)  s_period=[2, 4, 5, 7, 8, 10]  s_ME=[]  rho_ME=[]
   ( 29,  26)  s_period=[2, 4, 5, 7, 8, 10]  s_ME=[]  rho_ME=[]
   ( 29,  35)  s_period=[2, 3, 4, 5, 6, 7, 8, 9, 10]  s_ME=[]  rho_ME=[]
   ( 29,  39)  s_period=[2, 3, 4, 5, 6, 8, 9, 10]  s_ME=[]  rho_ME=[]
   ( 35,  26)  s_period=[2, 4, 5, 7, 8, 10]  s_ME=[]  rho_ME=[]
   ( 35,  29)  s_period=[2, 3, 4, 5, 6, 7, 8, 9, 10]  s_ME=[]  rho_ME=[]
   ( 35,  39)  s_period=[2, 3, 4, 5, 6, 8, 9, 10]  s_ME=[]  rho_ME=[]
   ( 39,  29)  s_period=[2, 3, 4, 5, 6, 8, 9, 10]  s_ME=[]  rho_ME=[]
   ( 39,  35)  s_period=[2, 3, 4, 5, 6, 8, 9, 10]  s_ME=[]  rho_ME=[]
-- pairing (150p, 150p) --
   ( 26,  29)  s_period=[2, 4, 5, 7, 8, 10]  s_ME=[]  rho_ME=[]
   ( 26,  35)  s_period=[2, 4, 5, 7, 8, 10]  s_ME=[]  rho_ME=[]
   ( 29,  26)  s_period=[2, 4, 5, 7, 8, 10]  s_ME=[]  rho_ME=[]
   ( 29,  35)  s_peri

   ( 29,  39)  s_period=[2, 3, 4, 5, 6, 8, 9, 10]  s_ME=[]  rho_ME=[]
   ( 35,  26)  s_period=[2, 4, 5, 7, 8, 10]  s_ME=[]  rho_ME=[]
   ( 35,  29)  s_period=[2, 3, 4, 5, 6, 7, 8, 9, 10]  s_ME=[]  rho_ME=[]
   ( 35,  39)  s_period=[2, 3, 4, 5, 6, 8, 9, 10]  s_ME=[]  rho_ME=[]
   ( 39,  29)  s_period=[2, 3, 4, 5, 6, 8, 9, 10]  s_ME=[]  rho_ME=[]
   ( 39,  35)  s_period=[2, 3, 4, 5, 6, 8, 9, 10]  s_ME=[]  rho_ME=[]

=== Group: close to 64  (k ∈ [65, 69]) ===

-- pairing (90p, 90p) --


   ( 65,  69)  s_period=[2, 3, 4, 5, 6, 8, 9, 10]  s_ME=[]  rho_ME=[]


   ( 69,  65)  s_period=[2, 3, 4, 5, 6, 8, 9, 10]  s_ME=[]  rho_ME=[]
-- pairing (150p, 150p) --


   ( 65,  69)  s_period=[2, 3, 4, 5, 6, 8, 9, 10]  s_ME=[]  rho_ME=[]


...

   (119, 113)  s_period=[2, 3, 4, 5, 6, 7, 8, 9, 10]  s_ME=[]  rho_ME=[]
-- pairing (90p, 150p) --


   (105, 113)  s_period=[2, 3, 4, 5, 6, 8, 9, 10]  s_ME=[]  rho_ME=[]


   (113, 105)  s_period=[2, 3, 4, 5, 6, 8, 9, 10]  s_ME=[]  rho_ME=[]


   (113, 119)  s_period=[2, 3, 4, 5, 6, 7, 8, 9, 10]  s_ME=[]  rho_ME=[]


   (119, 113)  s_period=[2, 3, 4, 5, 6, 7, 8, 9, 10]  s_ME=[]  rho_ME=[]


## Table 12 — verify *Period (≈ ρ)* and *Equidistribution* columns

The paper's Table 12 lists 49 combined CA-PRNGs with a claimed *Period*
exponent and *Equidistribution = ME*.  For each entry we compute:

- *period exponent*: with `ρ_i = 2^{k_i} − 1`, the effective component
  period at spacing `s` is `ρ_i / gcd(s, ρ_i)`; the combined period is
  the LCM, and we report `⌊log₂(combined period)⌋`.
- *ME verdict*: kernel's standard-convention `is_me()`.

Under the standard convention the paper's ME claim does not generally
hold — we surface the disagreement in the *paper ME* / *kernel ME*
columns.

In [8]:
TABLE12 = [
    # close-to-maximal period
    (31, 32,  7, 'max',     63),
    (31, 32,  8, 'max',     63),
    (31, 40,  8, 'max',     71),
    (35, 48,  8, 'max',     83),
    (41, 48,  8, 'max',     89),
    (43, 48,  8, 'max',     91),
    (47, 56,  8, 'max',    103),
    # reduced period
    (31, 32,  5, 'reduced', 61),
    (31, 32,  6, 'reduced', 62),
    (31, 32,  9, 'reduced', 62),
    (31, 32, 10, 'reduced', 61),
    (31, 40,  9, 'reduced', 70),
    (31, 40, 10, 'reduced', 69),
    (33, 40,  9, 'reduced', 72),
    (33, 40, 10, 'reduced', 71),
    (33, 56, 10, 'reduced', 87),
    (35, 48,  7, 'reduced', 81),
    (35, 48,  9, 'reduced', 80),
    (35, 48, 10, 'reduced', 81),
    (35, 64, 10, 'reduced', 97),
    (39, 40,  9, 'reduced', 78),
    (39, 40, 10, 'reduced', 77),
    (39, 56,  9, 'reduced', 94),
    (39, 56, 10, 'reduced', 93),
    (41, 48, 10, 'reduced', 87),
    (41, 64, 10, 'reduced',103),
    (43, 48,  9, 'reduced', 88),
    (43, 48, 10, 'reduced', 89),
    (43, 56, 10, 'reduced', 97),
    (45, 56,  9, 'reduced',100),
    (45, 56, 10, 'reduced', 99),
    (45, 64,  9, 'reduced',108),
    (45, 64, 10, 'reduced',107),
    (47, 56,  9, 'reduced',102),
    (47, 56, 10, 'reduced',101),
    (47, 64,  9, 'reduced',110),
    (47, 64, 10, 'reduced',109),
    (47, 72, 10, 'reduced',117),
    (51, 56,  9, 'reduced',106),
    (51, 56, 10, 'reduced',105),
    (53, 56,  9, 'reduced',108),
    (53, 56, 10, 'reduced',107),
    (55, 56,  9, 'reduced',110),
    (55, 56, 10, 'reduced',109),
    (59, 64, 10, 'reduced',121),
    (63, 64, 10, 'reduced',125),
    (63, 80, 10, 'reduced',141),
    (67, 72, 10, 'reduced',137),
    (71, 72, 10, 'reduced',141),
]

def combined_period_log2_ceil(k1, k2, s):
    """⌈log2(combined period at spacing s)⌉.

    Paper writes 'Period (≈ ρ) = 2^N' with N = ceil(log2(real period)) —
    e.g., (31, 32, 7) → period = (2^31−1)(2^32−1) ≈ 2^63 − 2^32, written
    as 2^63.
    """
    rho1 = 2**k1 - 1
    rho2 = 2**k2 - 1
    p1 = rho1 // math.gcd(s, rho1)
    p2 = rho2 // math.gcd(s, rho2)
    period = p1 * p2 // math.gcd(p1, p2)
    return period.bit_length()


print(f'{"k1":>3} {"k2":>3} {"s":>2}  {"kind":>7}  '
      f'{"paper exp":>9}  {"our exp":>7}  {"period ✓":>8}  '
      f'{"paper ME":>8}  {"kernel ME":>9}  {"ME ✓":>5}')
print('-' * 80)
period_ok = 0
me_agree = 0
for (k1, k2, s, kind, paper_exp) in TABLE12:
    our_exp = combined_period_log2_ceil(k1, k2, s)
    p_ok = (our_exp == paper_exp)
    period_ok += int(p_ok)
    _, _, comb = _make_combined(k1, CZ[str(k1)], k2, CZ[str(k2)], s)
    kernel_me = _run_equi(comb).is_me()
    paper_me = True
    me_agree += int(kernel_me == paper_me)
    print(f'{k1:>3} {k2:>3} {s:>2}  {kind:>7}  '
          f'{paper_exp:>9}  {our_exp:>7}  {("yes" if p_ok else "NO"):>8}  '
          f'{"ME":>8}  {("ME" if kernel_me else "NOT ME"):>9}  '
          f'{("yes" if kernel_me == paper_me else "NO"):>5}')

print(f'\nPeriod exponent agrees with paper:  {period_ok}/{len(TABLE12)}')
print(f'ME verdict agrees with paper:       {me_agree}/{len(TABLE12)}')

 k1  k2  s     kind  paper exp  our exp  period ✓  paper ME  kernel ME   ME ✓
--------------------------------------------------------------------------------
 31  32  7      max         63       63       yes        ME     NOT ME     NO
 31  32  8      max         63       63       yes        ME     NOT ME     NO
 31  40  8      max         71       71       yes        ME     NOT ME     NO
 35  48  8      max         83       83       yes        ME     NOT ME     NO
 41  48  8      max         89       89       yes        ME     NOT ME     NO
 43  48  8      max         91       91       yes        ME     NOT ME     NO
 47  56  8      max        103      103       yes        ME     NOT ME     NO
 31  32  5  reduced         61       61       yes        ME     NOT ME     NO
 31  32  6  reduced         62       62       yes        ME     NOT ME     NO
 31  32  9  reduced         62       62       yes        ME     NOT ME     NO
 31  32 10  reduced         61       61       yes        ME  

 51  56 10  reduced        105      105       yes        ME     NOT ME     NO
 53  56  9  reduced        108      108       yes        ME     NOT ME     NO
 53  56 10  reduced        107      107       yes        ME     NOT ME     NO
 55  56  9  reduced        110      110       yes        ME     NOT ME     NO
 55  56 10  reduced        109      109       yes        ME     NOT ME     NO
 59  64 10  reduced        121      121       yes        ME     NOT ME     NO
 63  64 10  reduced        125      125       yes        ME     NOT ME     NO
 63  80 10  reduced        141      141       yes        ME     NOT ME     NO
 67  72 10  reduced        137      137       yes        ME     NOT ME     NO
 71  72 10  reduced        141      141       yes        ME     NOT ME     NO

Period exponent agrees with paper:  49/49
ME verdict agrees with paper:       0/49


## Caveats

- The kernel implements the standard L'Écuyer / Proposition 2
  convention (`t · ℓ` rows, equi iff `rank = t · ℓ`).  The paper's
  ME claims in Tables 5, 6, 7, 8, 9, 10, 11, 12 are not in general
  reproducible under this convention; the kernel's verdicts here
  reflect the standard reading.
- Table 6 enumeration is gated by `MAX_K` (default 64).  Set
  `MAX_K = 128` to enumerate the paper's full range; the loop scales
  with `(MAX_K − 30)² × 9` equidistribution tests, each ≲ 20 ms.
- Adak-Das constructions for `k = 26, 29, 35, 39, 65, 69, 105, 113,
  119` are encoded directly from the rule (cell 0 vs cells 1..k−1) —
  the catalog stores only the size list, not per-cell positions, for
  these families.
- Combined `L = min(L_1, L_2)` with `L_i = min(k_i, 64)`; this is
  what `Combination._update_stats()` computes and what the per-`t`
  bounds in `Φ` reflect.